# PPO DOJO — kit retrain (GPU)

1. **Runtime → Change runtime type → T4 GPU** first (else this trains on CPU).
2. Edit `REPO` below to your repo URL, then **Runtime → Run all**.
3. Training runs in the background; the dashboard cell opens a live tab.
4. If the dashboard tab ever freezes, re-run the **fallback monitor** cell — it reads the same `metrics.jsonl` directly, no server needed.

In [ ]:
REPO = "https://github.com/exstronium-star/pvp-bot-colab"
ITERS = 1000
ENVS = 64
TICKS = 256
OUTDIR = "runs/kit1"
RESET_POOL = False  # True only if league snapshots teach cowardice; weights are kept either way

In [ ]:
!test -d dojo && (cd dojo && git pull --rebase) || git clone $REPO dojo
!pip install -q numpy  # torch ships with CUDA on colab; do NOT pip install torch

In [ ]:
!nvidia-smi -L || echo 'NO GPU VISIBLE - set Runtime -> Change runtime type -> T4 GPU'
!cd dojo && python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"
!cd dojo && python -m mc_pvp.validate 2>&1 | tail -3

In [ ]:
# training runs in the BACKGROUND so later cells (dashboard) can run.
# logs go to dojo/train_colab.log; checkpoints still save every 25 iters.
# --device auto picks cuda on this GPU. Old 46x10 checkpoints auto-upgrade
# to the 49x11 brain (webb + weave heads train fresh, the rest carries over).
import os
os.system('pkill -f "train.py --iters" 2>/dev/null')
cmd = (f"cd dojo && nohup python -u train.py --iters {ITERS} --envs {ENVS} --ticks {TICKS} "
       f"--outdir {OUTDIR} --resume --device auto" + (" --reset-pool" if RESET_POOL else "") +
       f" > train_colab.log 2>&1 & echo started pid $!")
print("+", cmd)
!{cmd}

In [ ]:
# peek at training progress anytime (re-run this cell)
!tail -n 8 dojo/train_colab.log
!ls -la dojo/runs/kit1/ 2>/dev/null | head

In [ ]:
# live dashboard: opens in a new tab. Polls every 2s with cache-bust,
# so it keeps updating instead of freezing after the first load.
# If the tab is empty/blocked, use the fallback monitor cell below instead.
!cd dojo && (pgrep -f "http.server 8000" >/dev/null || nohup python -m http.server 8000 --directory . >/dev/null 2>&1 &)
!sleep 1; curl -s -o /dev/null -w "local server: %{http_code} (want 200)\n" http://localhost:8000/tools/dashboard.html
from google.colab import output
output.serve_kernel_port_as_window(8000, path=f'/tools/dashboard.html?run={OUTDIR}')

In [ ]:
# fallback monitor: no server, no iframe - just re-run this cell anytime.
# This NEVER freezes: it re-reads metrics.jsonl fresh on every run.
import json, pathlib
mp = pathlib.Path(f'dojo/{OUTDIR}/metrics.jsonl')
rows = [json.loads(l) for l in mp.read_text().splitlines() if l.strip()] if mp.exists() else []
pts = [r for r in rows if 'ret' in r]
evs = [r for r in rows if r.get('event') == 'eval']
print(f'{len(rows)} points | {mp} exists={mp.exists()}')
if pts:
    t = pts[-1]
    print(f"iter {t['iter']} ret {t.get('ret')} ent {t.get('ent')} eps {t.get('eps')}")
    print(f"drills ret_melee {t.get('ret_melee')} ret_heal {t.get('ret_heal')} ret_web {t.get('ret_web')} ret_repot {t.get('ret_repot')}")
if evs:
    e = evs[-1]
    print(f"eval it {e['iter']}: win {e['win_rate']*100:.1f}% loss {e['loss_rate']*100:.1f}% dmg {e['dmg_dealt']:.1f}/{e['dmg_taken']:.1f}")
    for k in ('melee','heal','web','repot'):
        w = e.get(f'{k}_win')
        if w is not None:
            print(f"  {k}: win {w*100:.1f}% n={e.get(f'{k}_n')}")
if pts:
    import matplotlib.pyplot as plt
    xs = [r['iter'] for r in pts]
    plt.figure(); plt.plot(xs, [r.get('ret') for r in pts]); plt.title('avg reward'); plt.xlabel('iter'); plt.show()
    if evs:
        plt.figure(); plt.plot([r['iter'] for r in evs], [r['win_rate']*100 for r in evs]); plt.title('eval win %'); plt.xlabel('iter'); plt.show()
else:
    print('no data yet - training writes its first point after a few iters; check the tail cell for errors')

In [ ]:
# auto-watch: updates INSIDE this cell every 20s for ~10 min. No new tab,
# no proxy to freeze. Just leave it running and watch the numbers climb.
import json, pathlib, time
from IPython.display import clear_output
mp = pathlib.Path(f'dojo/{OUTDIR}/metrics.jsonl')
for _ in range(30):
    rows = [json.loads(l) for l in mp.read_text().splitlines() if l.strip()] if mp.exists() else []
    pts = [r for r in rows if 'ret' in r]
    evs = [r for r in rows if r.get('event') == 'eval']
    clear_output(wait=True)
    print(f'{len(rows)} points | {time.strftime("%H:%M:%S")}')
    if pts:
        t = pts[-1]
        print(f"iter {t['iter']} ret {t.get('ret')} ent {t.get('ent')}")
    if evs:
        e = evs[-1]
        print(f"eval it {e['iter']}: win {e['win_rate']*100:.1f}% dmg {e['dmg_dealt']:.1f}/{e['dmg_taken']:.1f}")
    time.sleep(20)

In [ ]:
# when training is done: pull the weights back to your PC
from google.colab import files
files.download(f'dojo/{OUTDIR}/ppo.pt')